# LSL Streams in xdf file

Author: Nicole Burke, PhD 


In [1]:
import pyxdf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
from scipy.io import wavfile


# Analyze the following streams for Graphomotor Test Run 


In [ ]:
# Load the XDF file
# >>>>>>>> UPDATE BELOW WHERE YOU SAVED THE DATA 
file_path = # UPDATE FILE PATH

streams, header = pyxdf.load_xdf(file_path)
# Print available keys in the 'info' dictionary for each stream
# for i, stream in enumerate(streams):
#     info_keys = list(stream['info'].keys())
#     print(f"Stream {i} info keys: {info_keys}")

# Custom functions
def print_column_names(stream):
        channels = stream['info']['desc'][0]['channels'][0]['channel']
        column_names = [channel['label'][0] for channel in channels]
        print(f"Column Names for: {column_names}")

def extract_single_stream(streams, stream_name):
    """Extract the time series and time stamps from a specified stream in xdf data."""
    for s in streams:
        if s['info']['name'][0] == stream_name:
            single_stream = s
            single_stream_time_series = np.array(single_stream['time_series'])
            # print("sample time series:", single_stream_time_series[:5])
            single_stream_time_stamps = np.array(single_stream['time_stamps'])
            # print("sample time stamps:", single_stream_time_stamps[:5])

            # Make dataframe
            single_stream_df = pd.DataFrame(single_stream_time_series)
            # Add timestamps
            single_stream_df["time_stamps"] = single_stream_time_stamps
    return single_stream_df, single_stream

def add_column_names(single_stream_df, single_stream):
    """Add column names to the single stream dataframes."""
    
    # Check if channel description exists and is valid
    try:
        if (single_stream['info']['desc'][0] is not None and 
            'channels' in single_stream['info']['desc'][0]):
            
            channels = single_stream['info']['desc'][0]['channels'][0]['channel']
            column_names = [channel['label'][0] for channel in channels]
            print('Column names from metadata:', column_names)
            
        else:
            raise KeyError("No valid channel description")
            
    except (KeyError, TypeError, IndexError):
        # Create column names if they aren't provided 
        channel_count = int(single_stream['info']['channel_count'][0])
        stream_name = single_stream['info']['name'][0]
        column_names = [f"{stream_name}_ch{i+1}" for i in range(channel_count)]
        print(f'Channel labels not available for {stream_name}. Using generic names:', column_names)

    # Apply the column names
    num_cols = len(column_names)
    single_stream_df.columns = column_names + list(single_stream_df.columns[num_cols:])

    return single_stream_df

Stream 3: Calculated effective sampling rate 85.1555 Hz is different from specified rate 1000.0000 Hz.


In [3]:
# Print information about the streams
print("-" * 40)
for stream in streams:
    print(f"Stream Name: {stream['info']['name'][0]}")
    print(f"Stream Type: {stream['info']['type'][0]}")
    print(f"Number of Channels: {stream['info']['channel_count'][0]}")
    print(f"Channel Format: {stream['info']['channel_format'][0]}")
    print(f"Sampling Rate: {stream['info']['nominal_srate'][0]}")
    print(f"Number of Samples: {len(stream['time_series'])}")
    # print("Sample Time Series Data:", stream[ 'time_series'][:5])
    # print("Sample Time Stamps:", stream['time_stamps'][:5])
    print("Dictionary Keys:", stream.keys())
    print("-"*40)


----------------------------------------
Stream Name: SceneCameraMarker
Stream Type: Markers
Number of Channels: 1
Channel Format: string
Sampling Rate: 0.000000000000000
Number of Samples: 2
Dictionary Keys: dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values'])
----------------------------------------
Stream Name: experiment_stream
Stream Type: Markers
Number of Channels: 1
Channel Format: int32
Sampling Rate: 0.000000000000000
Number of Samples: 123
Dictionary Keys: dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values'])
----------------------------------------
Stream Name: VideoMarkers
Stream Type: Markers
Number of Channels: 1
Channel Format: string
Sampling Rate: 0.000000000000000
Number of Samples: 2
Dictionary Keys: dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values'])
----------------------------------------
Stream Name: AudioMarkers
Stream Type: Markers
Number of Channe

### Extract Individual Data Streams

## **Note for Stefan**: You will need to update the inputs for the function 'extract_single_stream' to whatever the 'Stream Name' is above. This is what our system outputs, but this will vary depending on speicific devices you are using. 

In [ ]:
### EVENT MARKERS/PYGAME
print("-"*40)
print("Stream: EVENT MARKERS/PYGAME")
print("-"*40)
pygame_df, pygame = extract_single_stream(streams, "experiment_stream")
pygame_df = add_column_names(pygame_df, pygame)
print(pygame_df.head(5))

### AUDIO MARKERS
print("-"*40)
print("Stream: AUDIO MARKERS")
print("-"*40)
audio_markers_df, audio_markers_stream = extract_single_stream(streams, "AudioMarkers")
audio_markers_df = add_column_names(audio_markers_df, audio_markers_stream)
print(audio_markers_df.head(5))

### VideoMarkers
print("-"*40)
print("Stream: VIDEO MARKERS")
print("-"*40)
video_markers_df, video_markers_stream = extract_single_stream(streams, "VideoMarkers")
video_markers_df_markers_df = add_column_names(video_markers_df, video_markers_stream)
print(video_markers_df_markers_df.head(5))

### EYETRACKING
print("-"*40)
print("Stream: EYETRACKING")
print("-"*40)
eyetracking_df, eyetracking_stream = extract_single_stream(streams, 'Neon Companion_Neon Gaze')
eyetracking_df = add_column_names(eyetracking_df, eyetracking_stream)
print(eyetracking_df.head(5))

### NEON SCENE CAMERA (EYETRACKING MP4)
print("-"*40)
print("Stream: EYETRACKING SCENCE CAMERA")
print("-"*40)
eyetracking_camera_df, eyetracking_camera_stream = extract_single_stream(streams, 'SceneCameraMarker')
eyetracking_camera_df = add_column_names(eyetracking_camera_df, eyetracking_camera_stream)
print(eyetracking_camera_df.head(5))

### MINDLOGGER
print("-"*40)
print("Stream: MINDLOGGER")
print("-"*40)
mindlogger_df, mindlogger_stream = extract_single_stream(streams, "MindLogger")
mindlogger_df = add_column_names(mindlogger_df, mindlogger_stream)
print(mindlogger_df.head(5))

### BIOSIGNALS
print("-"*40)
print("Stream: BIOSIGNALS")
print("-"*40)
biosignals_df, biosignals_stream = extract_single_stream(streams, 'OpenSignals')
biosignals_df = add_column_names(biosignals_df, biosignals_stream)
print(biosignals_df.head(5))

### EEG 
print("-"*40)
print("Stream: EEG")
print("-"*40)
eeg_df, eeg_stream = extract_single_stream(streams, 'WS-default')
eeg_df = add_column_names(eeg_df, eeg_stream)
print(eeg_df.head(5))

### MoBIMarkers
print("-"*40)
print("Stream: MOBI MARKERS")
print("-"*40)
mobi_markers_df, mobi_markers = extract_single_stream(streams, "MobiMarkerStream")
mobi_markers_df = add_column_names(mobi_markers_df, mobi_markers)
print(mobi_markers_df.head(5))



## Event Markers 

In [ ]:
## Are event markers where they are supposed to be?

# Get rid of 0 event marker 
pygame_df = pygame_df[pygame_df['experiment_stream_ch1'] != 0]

# Duration of Pygame Data 
pygame_duration = (pygame_df['time_stamps'].max() - pygame_df['time_stamps'].min()) / 60
print(f"Duration of Pygame Data: {pygame_duration} mins")

# Plot
plt.figure()
plt.plot(pygame_df['time_stamps'], pygame_df['experiment_stream_ch1'], marker = "o")
plt.title("Event Markers x Time")
plt.ylabel("Event Markers")
plt.xlabel("Time Stamps")

# Annotate each event marker value on the plot
for idx, row in pygame_df.iterrows():
    plt.annotate(str(row['experiment_stream_ch1']),
                 (row['time_stamps'], row['experiment_stream_ch1']),
                 textcoords="offset points", xytext=(0,8), ha='center', fontsize=8)
plt.show()

pygame_df.sort_values('time_stamps', ascending=True)

## Eyetracking Data 

In [ ]:
######### EYETRACKING SAMPLING RATE

# Calculate time differences between consecutive samples
eyetracking_df['time_diff'] = eyetracking_df['time_stamps'].diff()

# Calculate sampling rate (Hz)
eyetracking_df['sampling_rate'] = 1 / eyetracking_df['time_diff']

# Display basic statistics
print("Sampling Rate Statistics:")
print(f"Mean sampling rate: {eyetracking_df['sampling_rate'].mean():.2f} Hz")
print(f"Std sampling rate: {eyetracking_df['sampling_rate'].std():.2f} Hz")
print(f"Min sampling rate: {eyetracking_df['sampling_rate'].min():.2f} Hz")
print(f"Max sampling rate: {eyetracking_df['sampling_rate'].max():.2f} Hz")

print("-"*40)
# print("timestamp min:", eyetracking_df['time_stamps'].min())
# print("timestamp max:", eyetracking_df['time_stamps'].max())
et_duration = (eyetracking_df['time_stamps'].max() - eyetracking_df['time_stamps'].min())/ 60
print(f"Duration of ET data: {et_duration} mins")
print("-"*40)

# Plot sampling rate over time as a line graph
plt.figure()
plt.plot(eyetracking_df['time_stamps'], eyetracking_df['sampling_rate'], linestyle='-', marker=None)
plt.title('Eyetracking Sampling Rate Over Time')
plt.ylabel('Sampling Rate (Hz)')
plt.xlabel('Time (seconds)')
plt.ylim(0, 220)
plt.show()


In [ ]:
######### EYETRACKING EXPLORATION

# Gaze position (x, y coordinates)
plt.figure(figsize=(7, 5))
plt.scatter(eyetracking_df['x'], eyetracking_df['y'], alpha=0.3, s=1)
plt.title('Gaze Position Heatmap')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.gca().set_aspect('equal')
plt.show()

# Gaze position over time
plt.figure(figsize=(10, 4))
plt.plot(eyetracking_df['time_stamps'], eyetracking_df['x'], label='X', alpha=0.7)
plt.plot(eyetracking_df['time_stamps'], eyetracking_df['y'], label='Y', alpha=0.7)
plt.title('Gaze Position Over Time')
plt.xlabel('Time (seconds)')
plt.ylabel('Position')
plt.legend()
plt.show()

# Data quality: Missing data over time
missing_x = eyetracking_df['x'].isna()
missing_y = eyetracking_df['y'].isna()

plt.figure(figsize=(10, 4))
plt.plot(eyetracking_df['time_stamps'], missing_x.astype(int), label='X missing', alpha=0.7)
plt.plot(eyetracking_df['time_stamps'], missing_y.astype(int), label='Y missing', alpha=0.7)
plt.title('Missing Data Over Time')
plt.xlabel('Time (seconds)')
plt.ylabel('Missing (1=Yes, 0=No)')
plt.legend()
plt.show()


## Biosignals

In [ ]:
### Biosignals sampling rate 

# Calcuate time difference between consecutive samples 
biosignals_df['time_diff'] = biosignals_df['time_stamps'].diff()

# Calculate sampling rate (Hz)
biosignals_df['sampling_rate'] = 1 / biosignals_df['time_diff']

# Display basic statistics
print("Sampling Rate Statistics:")
print(f"Mean sampling rate: {biosignals_df['sampling_rate'].mean():.2f} Hz")
print(f"Std sampling rate: {biosignals_df['sampling_rate'].std():.2f} Hz")
print(f"Min sampling rate: {biosignals_df['sampling_rate'].min():.2f} Hz")
print(f"Max sampling rate: {biosignals_df['sampling_rate'].max():.2f} Hz")

print("-"*40)
# print("timestamp min:", biosignals_df['time_stamps'].min())
# print("timestamp max:", biosignals_df['time_stamps'].max())
bio_duration = (biosignals_df['time_stamps'].max() - biosignals_df['time_stamps'].min()) / 60
print(f"Duration of Biosignals Data: {bio_duration} mins")

print("-"*40)

# Plot sampling rate over time as a line graph
plt.figure()
plt.plot(biosignals_df['time_stamps'], biosignals_df['sampling_rate'], linestyle='-', marker=None)
plt.title('Biosignals Sampling Rate Over Time')
plt.ylabel('Sampling Rate (Hz)')
plt.xlabel('Time (seconds)')
plt.ylim(900, 1100)
plt.show()


In [ ]:
######### BIOSIGNALS EXPLORATION
# Signal over time
# nSeq
plt.figure(figsize=(10, 4))
plt.plot(biosignals_df['time_stamps'], biosignals_df['nSeq'], label='signals', alpha=0.7)
plt.title('Signal Over Time: nSeq')
plt.xlabel('Time Stamps')
plt.ylabel('Signal')
plt.legend()
plt.show()

# RESPIRATION0
plt.figure(figsize=(10, 4))
plt.plot(biosignals_df['time_stamps'], biosignals_df['RIP0'], label='signals', alpha=0.7)
plt.title('Signal Over Time: RIP0')
plt.xlabel('Time Stamps')
plt.ylabel('Signal')
plt.legend()
plt.show()

# ECG
plt.figure(figsize=(10, 4))
plt.plot(biosignals_df['time_stamps'], biosignals_df['ECG1'], label='signals', alpha=0.7)
plt.title('Signal Over Time: ECG')
plt.xlabel('Time Stamps')
plt.ylabel('Signal')
plt.legend()
plt.show()

# EDA
plt.figure(figsize=(10, 4))
plt.plot(biosignals_df['time_stamps'], biosignals_df['EDA2'], label='signals', alpha=0.7)
plt.title('Signal Over Time: EDA')
plt.xlabel('Time Stamps')
plt.ylabel('Signal')
plt.legend()
plt.show()



## EEG 

In [ ]:
######### EEG SAMPLING RATE

# Calculate time differences between consecutive samples
eeg_df['time_diff'] = eeg_df['time_stamps'].diff()

# Calculate sampling rate (Hz)
eeg_df['sampling_rate'] = 1 / eeg_df['time_diff']

# Display basic statistics 
print("Sampling Rate Statistics:")
print(f"Mean sampling rate:, {eeg_df['sampling_rate'].mean():.2f} Hz")
print(f"std sampling rate: {eeg_df['sampling_rate'].std():.2f} Hz")
print(f"Min sampling rate: {eeg_df['sampling_rate'].min():.2f} Hz")
print(f"Max sampling rate: {eeg_df['sampling_rate'].max():.2f} Hz")
print("-"*40)
# print("timestamp min:", eeg_df['time_stamps'].min())
# print("timestamp max:", eeg_df['time_stamps'].max())
eeg_duration = (eeg_df['time_stamps'].max() - eeg_df['time_stamps'].min())/ 60
print(f"Duration of EEG data: {eeg_duration} mins")
print("-"*40)

# Plot sampling rate over time as a line graph 
plt.figure()
plt.plot(eeg_df['time_stamps'], eeg_df['sampling_rate'], linestyle = '-', marker = None)
plt.title("EEG Sampling Rate over time")
plt.xlabel("Time Stamps")
plt.ylabel("Sampling Rate (Hz)")
plt.ylim(250, 305)
plt.show()


In [ ]:
######### EEG EXPLORATION
# Define EEG and non-EEG (sensor) channels
eeg_channels = ['P3', 'C3', 'F3', 'Fz', 'F4', 'C4', 'P4', 'Cz', 'Pz',
                'Fp1', 'Fp2', 'T3', 'T5', 'O1', 'O2', 'F7', 'F8',
                'T6', 'T4', 'A2']
sensor_channels = ['X1', 'X2', 'X3', 'TRG', 'ACCT', 'ACCX', 'ACCY', 'ACCZ']
channels = eeg_channels + sensor_channels

# Set up the plot
plt.figure(figsize=(15, 6))

# Plot each channel
for ch in channels:
    missing = eeg_df[ch].isna().astype(int)
    color = None if ch in eeg_channels else 'lightgray'
    label = ch if ch in eeg_channels else None  # Only show EEG in legend
    plt.plot(eeg_df['time_stamps'], missing, label=label, alpha=0.8, linewidth=1.5, color=color)

# Format plot
plt.ylim(-1, 1.5)
plt.xlabel("Time")
plt.ylabel("Missing (1 = Yes, 0 = No)")
plt.title("Missing Data Over Time by Channel")
plt.legend(title="EEG Channels", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

plt.show()

## MindLogger

In [ ]:
df = mindlogger_df

# 🔹 Convert necessary columns to numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 🔹 Drop NaNs in x, y
df = df.dropna(subset=["x", "y"])

# 🔹 Ensure timestamps align with filtered data
# timestamps = timestamps[:len(df)]

# 🔹 Add timestamps to DataFrame
df["timestamp"] = df['time_stamps']

# 🔹 Normalize timestamps for slider values
df["time_index"] = np.arange(len(df))

# 🔹 Create Dash app
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Interactive X-Y Scatter Plot with Time Slider"),
    dcc.Graph(id="scatter-plot"),
    dcc.Slider(
        id="time-slider",
        min=df["time_index"].min(),
        max=df["time_index"].max(),
        value=df["time_index"].min(),
        marks={i: str(df["timestamp"].iloc[i])[:5] for i in range(0, len(df), len(df)//10)},  # Time labels
        step=1
    )
])

@app.callback(
    Output("scatter-plot", "figure"),
    [Input("time-slider", "value")]
)
def update_plot(selected_time_index):
    # 🔹 Filter data up to the selected time index
    filtered_df = df.iloc[:selected_time_index + 1]

    # 🔹 Get x and y range for auto-scaling
    x_min, x_max = filtered_df["x"].min(), filtered_df["x"].max()
    y_min, y_max = filtered_df["y"].min(), filtered_df["y"].max()

    # 🔹 Set 16:10 aspect ratio
    width = 800  # Adjust based on screen
    height = int(width * (10 / 16))  # Maintain 16:10 ratio

    # 🔹 Create scatter plot
    fig = px.scatter(
        filtered_df, 
        x="x", 
        y="y", 
        color="timestamp",
        color_continuous_scale="viridis",
        title="X-Y Position Over Time"
    )

    # 🔹 Fix aspect ratio dynamically
    fig.update_layout(
        width=width,  # Set width
        height=height,  # Maintain 16:10 aspect ratio
        xaxis=dict(range=[x_min, x_max]),  # Keep x-axis consistent
        yaxis=dict(range=[y_min, y_max], scaleanchor="x"),  # Lock aspect ratio
    )

    return fig

# 🔹 Run the app
if __name__ == "__main__":
    # app.run_server(debug=True)
    app.run(debug=True)


## MoBIMarkers

In [ ]:
# mobi_markers_df.to_csv('/Users/nicole.burke/OneDrive - Child Mind Institute/02_Projects/05_mobi_lab_build/08_xdf_processing/week 02.13.2026/sub_P5009941_02_06.csv')


